# CICA Atlas CORDEX EUR-11 rx1day bbox subset stalled job

When: 2026-10-28

The reported request subsets `c3s-cica-atlas.rx1day.CORDEX-EUR-11.rcp85.mon.v25` over bounding box `-25,34,40,72` (west, south, east, north). No time or time-component filter was supplied, so it covers the collection's full time range.

The job was reported as running for more than 2 hours 30 minutes. The service then reported: `Process failed: stalled-job recovery found no status update for at least 155 minutes.` This status describes the interval without an update; it does not establish the total processing time or the cause of the stall.

This notebook preserves the supplied request and status and reproduces the bbox subset using Rooki's Python workflow operators. The operator subsequently ran the 10-year reproduction; the supplied server diagnostic is recorded below. Code cells have no recorded outputs.


## Reported WPS status

```xml
<wps:Status creationTime="2026-10-28T17:41:04Z">
    <wps:ProcessFailed>
        <wps:ExceptionReport>
            <ows:Exception exceptionCode="NoApplicableCode" locator="None">
                <ows:ExceptionText>Process failed: stalled-job recovery found no status update for at least 155 minutes.</ows:ExceptionText>
            </ows:Exception>
        </wps:ExceptionReport>
    </wps:ProcessFailed>
</wps:Status>
```


## Original WPS workflow

The decoded JSON preserves the supplied CDATA payload, including the empty input name, `inputs/` reference, and `subset__1` step name.


In [1]:
request = {
    "inputs": {"": ["c3s-cica-atlas.rx1day.CORDEX-EUR-11.rcp85.mon.v25"]},
    "steps": {
        "subset__1": {
            "run": "subset",
            "in": {"collection": "inputs/", "area": "-25,34,40,72"},
        }
    },
    "outputs": {"output": "subset__1/output"},
    "doc": "workflow",
}

request


{'inputs': {'': ['c3s-cica-atlas.rx1day.CORDEX-EUR-11.rcp85.mon.v25']},
 'steps': {'subset__1': {'run': 'subset',
   'in': {'collection': 'inputs/', 'area': '-25,34,40,72'}}},
 'outputs': {'output': 'subset__1/output'},
 'doc': 'workflow'}

## Build the Python workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` to test another deployment. The empty input name ensures the serialized Python workflow matches the original request exactly.


In [2]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [3]:
ds = ops.Input("", request["inputs"][""])
workflow = ops.Subset(ds, area=request["steps"]["subset__1"]["in"]["area"])

serialized_request = json.loads(workflow._serialise())
assert serialized_request == request
serialized_request


{'inputs': {'': ['c3s-cica-atlas.rx1day.CORDEX-EUR-11.rcp85.mon.v25']},
 'steps': {'subset__1': {'run': 'subset',
   'in': {'collection': 'inputs/', 'area': '-25,34,40,72'}}},
 'outputs': {'output': 'subset__1/output'},
 'doc': 'workflow'}

## Time a one-year bbox subset

Original file: `rx1day_CORDEX-EUR-11_rcp85_mon_200601-210012_v025.nc` — reported size **33G**, covering January 2006–December 2100 (**95 years**).

The operator reports that a one-year slice works. Run the same bbox subset for January–December 2006, then multiply its runtime by **95** for a rough full-range guess. Runtime may not scale linearly, and this cannot predict a stalled job.


In [4]:
from time import perf_counter

sample_workflow = ops.Subset(ds, area="-25,34,40,72", time="2006-01/2006-12")

start = perf_counter()
sample_response = sample_workflow.orchestrate()
elapsed = perf_counter() - start

print(f"One-year subset: {elapsed / 60:.1f} minutes")
print(f"Succeeded: {sample_response.ok}")
print(f"Status: {sample_response.status}")


One-year subset: 1.7 minutes
Succeeded: True
Status: ProcessSucceeded


In [5]:
if sample_response.ok:
    estimated = elapsed * 95
    print(f"Full-range guess (2006–2100): {estimated / 3600:.2f} hours")


Full-range guess (2006–2100): 2.63 hours


The original full-range `workflow` is built above. Run `workflow.orchestrate()` separately when needed; the cells here only submit the one-year subset.


## One-year output

Reported output for the 2006 bbox subset:

```text
253M    rx1day_CORDEX-EUR-11_rcp85_mon_20060101-20061201.nc
```

Scaling the reported disk usage by 95 gives roughly **23.5 GiB** for the complete 2006–2100 bbox output, assuming similar output size per year. This is a size estimate; the runtime guess still needs the measured one-year duration.


## 10-year reproduction: segmentation fault (2026-10-29)

The operator reports that the first one-year batch (2006) was written before the job died. The log shows batch 2 (2007) starting, then the launcher exiting with `Segmentation fault (core dumped)` (Slurm job `40212`, PID `157389`). The 10-year subset did not complete, so it provides no successful runtime to scale to 95 years.

The batching plan selected ten one-year batches under a configured memory limit of 4,244,000,000 bytes, with estimated batch memory of 2,449,998,144 bytes. The recorded RSS values are snapshots, not peak memory measurements; this excerpt does not establish an out-of-memory kill or the cause of the segmentation fault.

The multiple-fill-value warnings occur before both batches. The leaked-semaphore warning appears at shutdown. Neither warning establishes the crash cause. The next useful evidence is the core dump/backtrace and Slurm peak-memory/exit accounting.

### Supplied server log

```text
/usr/local/anaconda/envs/rook/lib/python3.14/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'rx1day' has multiple fill values {np.float32(1.0384594e+34), np.float32(-1.7014118e+38)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
Planning subset batching for 1 source(s): requested_time=2006-01-01T00:00:00/2015-12-31T23:59:59
/usr/local/anaconda/envs/rook/lib/python3.14/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'rx1day' has multiple fill values {np.float32(1.0384594e+34), np.float32(-1.7014118e+38)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
[diagnostic] 2026-10-29T14:28:15Z pid=157389 VmRSS: 351768 kB Woodpecker fixes applied | dataset=c3s-cica-atlas.rx1day.CORDEX-EUR-11.rcp85.mon.v25 recipe=c3s.atlas phase=apply attempted=2 changed=2
Subset batching plan for c3s-cica-atlas.rx1day.CORDEX-EUR-11.rcp85.mon.v25: calendar=standard, timesteps_per_year=12, estimated_selected_timesteps=None, configured_target_timesteps=2000, memory_limit_bytes=4244000000, bytes_per_timestep=102083256, memory_target_timesteps=20, effective_target_timesteps=20, estimated_batch_memory_bytes=2449998144, batch_size=1 years, batches=10
Running subset batch 1/10 for c3s-cica-atlas.rx1day.CORDEX-EUR-11.rcp85.mon.v25: 2006-01-01T00:00:00/2006-12-31T23:59:59 using 1 source file(s)
/usr/local/anaconda/envs/rook/lib/python3.14/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'rx1day' has multiple fill values {np.float32(1.0384594e+34), np.float32(-1.7014118e+38)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
[diagnostic] 2026-10-29T14:28:15Z pid=157389 VmRSS: 351396 kB Woodpecker fixes applied | dataset=c3s-cica-atlas.rx1day.CORDEX-EUR-11.rcp85.mon.v25 recipe=c3s.atlas phase=apply attempted=2 changed=2
Running subset batch 2/10 for c3s-cica-atlas.rx1day.CORDEX-EUR-11.rcp85.mon.v25: 2007-01-01T00:00:00/2007-12-31T23:59:59 using 1 source file(s)
/usr/local/anaconda/envs/rook/lib/python3.14/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'rx1day' has multiple fill values {np.float32(1.0384594e+34), np.float32(-1.7014118e+38)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
[diagnostic] 2026-10-29T14:29:46Z pid=157389 VmRSS: 475456 kB Woodpecker fixes applied | dataset=c3s-cica-atlas.rx1day.CORDEX-EUR-11.rcp85.mon.v25 recipe=c3s.atlas phase=apply attempted=2 changed=2
/usr/local/anaconda/envs/rook/lib/python3.14/multiprocessing/resource_tracker.py:475: UserWarning: resource_tracker: There appear to be 1 leaked semaphore objects to clean up at shutdown: {'/mp-g58ti5im'}
  warnings.warn(
/var/spool/slurm/d/job40212/slurm_script: line 2: 157389 Segmentation fault      (core dumped) /usr/local/Miniforge3-24.9.2-0-Linux-x86_64/envs/rook/bin/joblauncher '-c' '/var/lib/pywps/tmp/rook/pywps_process_1e2xfqn0/pywps.cfg' '/var/lib/pywps/tmp/rook/pywps_process_1e2xfqn0/job_fi5qspud.dump'
```
